In [1]:
# ========== 导入库 ==========
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import warnings
warnings.filterwarnings('ignore')

from vmdpy import VMD
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import scipy.stats as stats
import statsmodels.api as sm

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi']  = 300

In [3]:
# ========== 1. 辅助函数 ==========

def ljung_box_q(series, lags=10):
    """计算Ljung-Box Q统计量"""
    result = sm.stats.acorr_ljungbox(series, lags=[lags], return_df=True)
    return result['lb_stat'].values[0]

def descriptive_stats_table(series, name):
    """生成单个序列的描述统计表（行：统计量）"""
    stats_dict = {
        'Count': len(series),
        'Mean': series.mean(),
        'Std': series.std(),
        'Skewness': stats.skew(series),
        'Excess Kurtosis': stats.kurtosis(series),
        'Jarque-Bera': stats.jarque_bera(series)[0],
        'Q(10)': ljung_box_q(series)
    }
    return pd.DataFrame([stats_dict], index=[name]).round(6)

def latex_triple_table(df, caption, label, float_format='%.6f'):
    """输出三线表格式的 LaTeX 代码"""
    latex = df.to_latex(float_format=float_format, escape=False)
    
    latex = latex.replace('\\begin{tabular}', '\\begin{tabular}{' + 'c'*len(df.columns) + '}')
    lines = latex.split('\n')
    lines.insert(1, '\\toprule')
    lines.insert(3, '\\midrule')
    lines.append('\\bottomrule')
    return '\n'.join(lines)

In [5]:
# ========== 2. VMD 分解与可视化 ==========

def vmd_decompose(signal, K=5, alpha=2000):
    """VMD分解，返回DataFrame（时间序列索引）"""
    signal = signal.copy().astype(np.float64)
    original_len = len(signal)
    if original_len % 2 == 1:
        signal = np.append(signal, signal[-1])
    u, _, _ = VMD(signal, alpha, 0, K, 0, 1, 1e-7)
    if u.shape[0] == K:
        imfs = [u[i, :original_len] for i in range(K-1)]
        residue = u[-1, :original_len]
    else:
        imfs = [u[:original_len, i] for i in range(K-1)]
        residue = u[:original_len, -1]
    components = np.vstack((imfs, residue)).T
    columns = [f'IMF{i+1}' for i in range(len(imfs))] + ['Res']
    decompose_df = pd.DataFrame(components, columns=columns)
    return decompose_df

In [7]:
# ========== 3. 模型运行函数 ==========

def run_model(data_file, dataset_name):
    print(f"\n{'='*60}")
    print(f"正在处理数据集: {dataset_name} (文件: {data_file})")
    print('='*60)

    # ---------- 3.1 数据读取与 RV 计算 ----------
    temp_data = pd.read_csv(data_file)
    
    if 'trading_date' in temp_data.columns:
        date_col = 'trading_date'
    else:
        date_col = 'date'
    
    total_days = sorted(set(temp_data[date_col]), key=list(temp_data[date_col]).index)
    daily_close = []
    for day in total_days:
        day_data = temp_data[temp_data[date_col] == day]
        daily_close.append(day_data.iloc[-1]['close'])

    data = pd.DataFrame({
        'date': pd.to_datetime(total_days),
        'close': daily_close
    })
    data.set_index('date', inplace=True)

    data = data.sort_index()

    data['log return'] = np.log(data['close'] / data['close'].shift(1))
    dfr = data['log return'].copy()
    dfrv = dfr.groupby(pd.Grouper(freq='1d')).apply(lambda x: np.sqrt(np.sum(x**2)))
    data['RV'] = dfrv
    data.replace(np.inf, 0, inplace=True)
    RVs = data['RV'].dropna().copy()

    print(f"RV 序列长度: {len(RVs)}")
    print(f"RV 均值: {RVs.mean():.6f}, 标准差: {RVs.std():.6f}")

    # ---------- 3.2 Figure 1 ----------
    fig1, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(RVs.index, RVs, color='black', linewidth=1)
    ax1.set_title(f'Original Volatility Series ({dataset_name})')
    ax1.set_ylabel('RV')
    ax1.set_xlabel('Date')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    fig1.savefig(f'Figure1_RV_series_{dataset_name}.png', dpi=300)
    plt.close()
    print(f"Figure1 已保存: Figure1_RV_series_{dataset_name}.png")

    # ---------- 3.3 VMD 分解 ----------
    print("正在进行 VMD 分解...")
    decompose_df = vmd_decompose(RVs, K=5, alpha=2000)
    decompose_df.index = RVs.index
    print(f"分解得到 {decompose_df.shape[1]} 个分量: {list(decompose_df.columns)}")

    # ---------- 3.4 Figure 2 ----------
    n_components = decompose_df.shape[1]
    fig2, axes2 = plt.subplots(n_components + 1, 1, figsize=(12, 2*(n_components+1)), sharex=True)

    axes2[0].plot(RVs.index, RVs, linewidth=0.8, color='black')
    axes2[0].set_ylabel('RV')
    axes2[0].set_title(f'Original RV Series ({dataset_name})')

    for i, col in enumerate(decompose_df.columns):
        axes2[i+1].plot(decompose_df.index, decompose_df[col], linewidth=0.5, color='b')
        axes2[i+1].set_ylabel(col)

    axes2[-1].set_xlabel('Date')
    axes2[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    fig2.suptitle(f'VMD Decomposition Results ({dataset_name})', fontsize=14)
    plt.tight_layout()
    fig2.savefig(f'Figure2_VMD_decomposition_{dataset_name}.png', dpi=300)
    plt.close()
    print(f"Figure2 已保存: Figure2_VMD_decomposition_{dataset_name}.png")

    # ---------- 3.5 Table 1 & Table 2 ----------
    stats_rv = descriptive_stats_table(RVs, 'RV')
    stats_rv.to_csv(f'Table1_RV_stats_{dataset_name}.csv', index=False)

    print(f"Table1 已保存: Table1_RV_stats_{dataset_name}.csv")
    print("\n===== LaTeX 三线表: Table1 (RV) =====")
    print(latex_triple_table(
        stats_rv.T,
        caption=f"Descriptive statistics of RV series ({dataset_name})",
        label=f"tab:rv_stats_{dataset_name}"
    ))

    stats_comp = []
    for col in decompose_df.columns:
        stats_comp.append(descriptive_stats_table(decompose_df[col], col))

    stats_vmd = pd.concat(stats_comp, axis=0)
    stats_vmd.to_csv(f'Table2_VMD_stats_{dataset_name}.csv', index=False)

    print(f"Table2 已保存: Table2_VMD_stats_{dataset_name}.csv")
    print("\n===== LaTeX 三线表: Table2 (VMD components) =====")
    print(latex_triple_table(
        stats_vmd,
        caption=f"Descriptive statistics of VMD components ({dataset_name})",
        label=f"tab:vmd_stats_{dataset_name}"
    ))

    # ---------- 3.6 时间序列划分 ----------
    def TimeSeries(dataset, start_index, history_size, end_index, step, target_size, point_time, true):
        data, labels = [], []
        start_index = start_index + history_size

        if end_index is None:
            end_index = len(dataset) - target_size

        for i in range(start_index, end_index):
            index = range(i - history_size, i, step)
            data.append(dataset.iloc[index].values)

            if point_time:
                labels.append(true.iloc[i + target_size])
            else:
                labels.append(true.iloc[i:i + target_size].values)

        return np.array(data), np.array(labels)

    def get_tain_val_test(serie_data, window_size):
        train_num = int(len(serie_data) * 0.7)
        val_num = int(len(serie_data) * 0.9)

        history_size = window_size
        target_size = 0
        step = 1

        x_train, y_train = TimeSeries(
            serie_data, 0, history_size, train_num,
            step, target_size, True, serie_data
        )

        x_val, y_val = TimeSeries(
            serie_data, train_num, history_size, val_num,
            step, target_size, True, serie_data
        )

        x_test, y_test = TimeSeries(
            serie_data, val_num, history_size, len(serie_data),
            step, target_size, True, serie_data
        )

        return x_train, y_train, x_val, y_val, x_test, y_test

    window_size = 26

    # ---------- 3.7 GRU 模型 ----------
    def build_gru(input_shape):
        model = Sequential([
            GRU(30, return_sequences=False, input_shape=input_shape),
            Dense(1)
        ])
        model.compile(loss='mse', optimizer='adam')
        return model

    def train_and_predict_all(model_builder, x_tr, y_tr, x_v, y_v, x_te, epochs=15, batch_size=16):
        input_shape = (x_tr.shape[1], 1)
        model = model_builder(input_shape)

        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        model.fit(
            x_tr, y_tr,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(x_v, y_v),
            verbose=0,
            callbacks=[early_stop]
        )

        y_tr_pred = model.predict(x_tr, verbose=0).flatten()
        y_v_pred = model.predict(x_v, verbose=0).flatten()
        y_te_pred = model.predict(x_te, verbose=0).flatten()

        return y_tr_pred, y_v_pred, y_te_pred

    epochs = 15
    batch_size = 16

    train_preds, val_preds, test_preds = [], [], []

    for col in decompose_df.columns:
        print(f"处理分量: {col}")

        serie = decompose_df[col]

        x_tr, y_tr, x_v, y_v, x_te, y_te = get_tain_val_test(serie, window_size)

        x_tr = x_tr.reshape(x_tr.shape[0], x_tr.shape[1], 1)
        x_v = x_v.reshape(x_v.shape[0], x_v.shape[1], 1)
        x_te = x_te.reshape(x_te.shape[0], x_te.shape[1], 1)

        y_tr_pred, y_v_pred, y_te_pred = train_and_predict_all(
            build_gru,
            x_tr, y_tr,
            x_v, y_v,
            x_te,
            epochs,
            batch_size
        )

        train_preds.append(y_tr_pred)
        val_preds.append(y_v_pred)
        test_preds.append(y_te_pred)

    # ---------- 3.8 Figure 3 ----------
    comp_pred_full = []

    for i in range(len(decompose_df.columns)):
        pred_concat = np.concatenate([train_preds[i], val_preds[i], test_preds[i]])
        comp_pred_full.append(pred_concat)

    true_comp_full = [
        decompose_df[col].iloc[window_size:].values
        for col in decompose_df.columns
    ]

    min_len = min(len(p) for p in comp_pred_full)

    fig3, axes3 = plt.subplots(n_components, 1, figsize=(12, 2*n_components), sharex=True)

    for i, col in enumerate(decompose_df.columns):
        true_vals = true_comp_full[i][:min_len]
        pred_vals = comp_pred_full[i][:min_len]

        axes3[i].plot(true_vals, label='True', color='black', linewidth=0.8)
        axes3[i].plot(pred_vals, label='Predicted', color='red', alpha=0.8)
        axes3[i].set_ylabel(col)
        axes3[i].legend(loc='upper right', fontsize='small')

    axes3[-1].set_xlabel('Time steps')
    fig3.suptitle(f'GRU Predictions on Each VMD Component ({dataset_name})', fontsize=14)
    plt.tight_layout()
    fig3.savefig(f'Figure3_Component_predictions_{dataset_name}.png', dpi=300)
    plt.close()

    print(f"Figure3 已保存: Figure3_Component_predictions_{dataset_name}.png")

    # ---------- 3.9 随机森林重构 ----------
    min_len_train = min(len(p) for p in train_preds)
    min_len_val = min(len(p) for p in val_preds)
    min_len_test = min(len(p) for p in test_preds)

    train_pred_matrix = np.array([p[:min_len_train] for p in train_preds]).T
    val_pred_matrix = np.array([p[:min_len_val] for p in val_preds]).T
    test_pred_matrix = np.array([p[:min_len_test] for p in test_preds]).T

    train_num = int(len(RVs) * 0.7)
    val_num = int(len(RVs) * 0.9)

    y_train_true = RVs.iloc[window_size:train_num].values
    y_val_true = RVs.iloc[train_num + window_size:val_num].values
    y_test_true = RVs.iloc[val_num + window_size:].values

    min_len_train = min(train_pred_matrix.shape[0], len(y_train_true))
    min_len_val = min(val_pred_matrix.shape[0], len(y_val_true))
    min_len_test = min(test_pred_matrix.shape[0], len(y_test_true))

    train_pred_matrix = train_pred_matrix[:min_len_train]
    val_pred_matrix = val_pred_matrix[:min_len_val]
    test_pred_matrix = test_pred_matrix[:min_len_test]

    y_train_true = y_train_true[:min_len_train]
    y_val_true = y_val_true[:min_len_val]
    y_test_true = y_test_true[:min_len_test]

    rf = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(train_pred_matrix, y_train_true)

    final_pred_train = rf.predict(train_pred_matrix)
    final_pred_val = rf.predict(val_pred_matrix)
    final_pred = rf.predict(test_pred_matrix)

    # ---------- 3.10 评估指标 ----------
    def hmae(y_true, y_pred):
        return mean_absolute_error(y_true, y_pred) / np.mean(y_pred)

    def hmse(y_true, y_pred):
        return mean_squared_error(y_true, y_pred) / (np.mean(y_pred) ** 2)

    mae = mean_absolute_error(y_test_true, final_pred)
    mse = mean_squared_error(y_test_true, final_pred)
    r2 = r2_score(y_test_true, final_pred)
    hmae_val = hmae(y_test_true, final_pred)
    hmse_val = hmse(y_test_true, final_pred)

    print(f"\n========== {dataset_name} 最终结果 ==========")
    print("注意：当前结果的最终评估目标已经修正为原始 RV。")
    print(f"MAE  = {mae:.6f}")
    print(f"MSE  = {mse:.6f}")
    print(f"R²   = {r2:.4f}")
    print(f"HMAE = {hmae_val:.6f}")
    print(f"HMSE = {hmse_val:.6f}")

    results_df = pd.DataFrame({
        'Dataset': [dataset_name],
        'MAE': [mae],
        'MSE': [mse],
        'R2': [r2],
        'HMAE': [hmae_val],
        'HMSE': [hmse_val]
    })

    results_df.to_csv(f'Table4_Model_evaluation_{dataset_name}.csv', index=False)
    print(f"Table4 (单个数据集) 已保存: Table4_Model_evaluation_{dataset_name}.csv")

    # ---------- 3.11 Figure 4 ----------
    fig4, ax4 = plt.subplots(figsize=(14, 5))

    test_idx = RVs.index[val_num + window_size : val_num + window_size + len(final_pred)]

    ax4.plot(test_idx, y_test_true, label='True RV', color='black', linewidth=1.2)
    ax4.plot(test_idx, final_pred, label='Predicted (VMD+GRU+RF)', color='red', alpha=0.8)

    ax4.legend()
    ax4.set_title(f'Final Volatility Forecasting Results ({dataset_name})')
    ax4.set_xlabel('Date')
    ax4.set_ylabel('RV')
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    plt.xticks(rotation=45)
    plt.tight_layout()
    fig4.savefig(f'Figure4_Final_prediction_{dataset_name}.png', dpi=300)
    plt.close()

    print(f"Figure4 已保存: Figure4_Final_prediction_{dataset_name}.png")

    return {
        'dataset': dataset_name,
        'MAE': mae,
        'MSE': mse,
        'R2': r2,
        'HMAE': hmae_val,
        'HMSE': hmse_val,
        'y_true': y_test_true,
        'y_pred': final_pred,
        'train_true': y_train_true,
        'train_pred': final_pred_train,
        'val_true': y_val_true,
        'val_pred': final_pred_val
    }

In [9]:
# ========== 4. 主程序：遍历数据集并汇总 ==========

datasets = [
    ('P888.csv', 'P888'),
    ('CSI300.csv', 'CSI300')
]

all_results = []
for file, name in datasets:
    if not os.path.exists(file):
        print(f"警告: 文件 {file} 不存在，跳过")
        continue
    res = run_model(file, name)
    all_results.append(res)

# 汇总表格 (Table 4)
print("\n" + "="*60)
print("所有数据集结果汇总 (Table 4)")
print("="*60)
summary_df = pd.DataFrame([{
    'Dataset': r['dataset'],
    'MAE': r['MAE'],
    'MSE': r['MSE'],
    'R2': r['R2'],
    'HMAE': r['HMAE'],
    'HMSE': r['HMSE']
} for r in all_results])
print(summary_df.round(6))
summary_df.to_csv('Table4_Model_evaluation_all.csv', index=False)
print("\n汇总结果已保存至 Table4_Model_evaluation_all.csv")
print("\n===== LaTeX 三线表: Table4 =====")
print(latex_triple_table(summary_df.round(6), caption="Forecasting performance comparison", label="tab:model_eval"))

# Table 3: 超参数设置
hyper_df = pd.DataFrame({
    'Parameter': ['VMD modes (K)', 'VMD alpha', 'Window size', 'GRU units', 'Epochs', 'Batch size', 'RF trees', 'Random seed'],
    'Value': [5, 2000, 26, 30, 15, 16, 100, 42]
})
print("\n===== Table 3: Hyperparameters =====")
print(hyper_df)
hyper_df.to_csv('Table3_Hyperparameters.csv', index=False)
print("Table3 已保存: Table3_Hyperparameters.csv")
print("\n===== LaTeX 三线表: Table3 =====")
print(latex_triple_table(hyper_df, caption="Hyperparameter settings of VMD-GRU-RF", label="tab:hyper"))


正在处理数据集: P888 (文件: P888.csv)
RV 序列长度: 2917
RV 均值: 0.056880, 标准差: 0.150164
Figure1 已保存: Figure1_RV_series_P888.png
正在进行 VMD 分解...
分解得到 5 个分量: ['IMF1', 'IMF2', 'IMF3', 'IMF4', 'Res']
Figure2 已保存: Figure2_VMD_decomposition_P888.png
Table1 已保存: Table1_RV_stats_P888.csv

===== LaTeX 三线表: Table1 (RV) =====
\begin{tabular}{c}{lr}
\toprule
\toprule
\midrule
 & RV \\
\midrule
Count & 2917.000000 \\
Mean & 0.056880 \\
Std & 0.150164 \\
Skewness & 9.396408 \\
Excess Kurtosis & 129.283608 \\
Jarque-Bera & 2074402.812833 \\
Q(10) & 5319.213671 \\
\bottomrule
\end{tabular}

\bottomrule
Table2 已保存: Table2_VMD_stats_P888.csv

===== LaTeX 三线表: Table2 (VMD components) =====
\begin{tabular}{ccccccc}{lrrrrrrr}
\toprule
\toprule
\midrule
 & Count & Mean & Std & Skewness & Excess Kurtosis & Jarque-Bera & Q(10) \\
\midrule
IMF1 & 2917 & 0.056108 & 0.096479 & 4.442929 & 23.698030 & 77854.135946 & 27387.563956 \\
IMF2 & 2917 & 0.000035 & 0.039709 & 0.793826 & 99.014817 & 1191892.841666 & 11876.903461 \\
IMF3 